In [1]:
from transformers import pipeline

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
classifier = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10466.49it/s]


In [3]:
text = "The food was delicious and the staff was very friendly."

result = classifier(text, truncation=True)

print(result)

[{'label': '4 stars', 'score': 0.45791590213775635}]


In [9]:
import pandas as pd

df = pd.read_csv("../data/yelp_sample_1000.csv")

pilot_df = df.groupby("stars").sample(n=5, random_state=42)

In [10]:
print(pilot_df.shape)
print(pilot_df["stars"].value_counts().sort_index())

(25, 3)
stars
1    5
2    5
3    5
4    5
5    5
Name: count, dtype: int64


In [11]:
texts = pilot_df["text"].tolist()

predictions = classifier(
    texts,
    truncation=True,
    batch_size=8
)

print(predictions[:5])

[{'label': '2 stars', 'score': 0.4985164403915405}, {'label': '1 star', 'score': 0.730534017086029}, {'label': '1 star', 'score': 0.33332544565200806}, {'label': '2 stars', 'score': 0.38497018814086914}, {'label': '1 star', 'score': 0.8629486560821533}]


In [13]:
results_df = pilot_df[["stars", "text"]].reset_index(drop=True).copy()

results_df["predicted_stars"] = [
    int(pred["label"].split()[0])
    for pred in predictions
]

results_df["score"] = [
    pred["score"]
    for pred in predictions
]

In [14]:
print(results_df[["stars", "predicted_stars", "score"]].to_string(index=False))

 stars  predicted_stars    score
     1                2 0.498516
     1                1 0.730534
     1                1 0.333325
     1                2 0.384970
     1                1 0.862949
     2                2 0.718386
     2                3 0.301070
     2                3 0.432631
     2                2 0.494636
     2                3 0.441935
     3                4 0.249133
     3                5 0.568392
     3                4 0.575782
     3                2 0.435237
     3                2 0.533948
     4                3 0.537191
     4                4 0.958952
     4                4 0.313210
     4                4 0.604127
     4                5 0.504682
     5                5 0.786859
     5                5 0.717740
     5                4 0.549697
     5                4 0.511228
     5                5 0.934434


In [15]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7414.21it/s]


In [16]:
candidate_labels = [
    "a very negative customer review",
    "a negative customer review",
    "a mixed or neutral customer review",
    "a positive customer review",
    "a very positive customer review"
]

In [17]:
text = "The service was terrible and the food was cold."

result = zero_shot(
    text,
    candidate_labels=candidate_labels,
    multi_label=False
)

print(result)

{'sequence': 'The service was terrible and the food was cold.', 'labels': ['a negative customer review', 'a very negative customer review', 'a mixed or neutral customer review', 'a positive customer review', 'a very positive customer review'], 'scores': [0.4372319281101227, 0.32983970642089844, 0.17932017147541046, 0.029056517407298088, 0.024551648646593094]}


In [21]:
pilot2_df = pilot_df.groupby("stars").sample(n=2, random_state=42)

texts = pilot2_df["text"].tolist()

predictions = zero_shot(
    texts,
    candidate_labels=candidate_labels,
    multi_label=False
)

print(len(predictions))
print(predictions[0])

10
{'sequence': 'Buckhead Saloon; The Worst Jack-hole in Charlotte, North Carolina.\\n\\nBetween the overpriced bottled beer past its expiration date to the broken glass and urine on the floor to the snooty effeminate manager, this place should be avoided at all costs, unless your goal is to catch hepatitis. \\nThe one time prior I visited this establishment I was let in with no cover even though it was relatively early, because as the guy at the door stated \\"there are not enough good looking guys in there\\". The women were leaving in herds, so I walked in and saw exactly why. This place is what is wrong with the universe. Even if there is a semi-decent acoustic band playing, you would never hear them over the drunken squall of pseudo-urban rednecks that are just happy to be breathing. This is the type of place that would brag about being a meat market if it wasn\'t so wretched. Don\'t just take my word for it. Read the reviews by the \\"guys\\" who gave it 4 stars and you will unde

In [22]:
zero_shot_results = pd.DataFrame({
    "stars": pilot2_df["stars"].values,
    "zero_shot_label": [pred["labels"][0] for pred in predictions],
    "score": [pred["scores"][0] for pred in predictions]
})

print(zero_shot_results.to_string(index=False))

 stars                    zero_shot_label    score
     1         a positive customer review 0.201998
     1 a mixed or neutral customer review 0.659484
     2    a very negative customer review 0.235896
     2 a mixed or neutral customer review 0.269301
     3         a positive customer review 0.222450
     3    a very positive customer review 0.407212
     4    a very positive customer review 0.594614
     4 a mixed or neutral customer review 0.219253
     5         a positive customer review 0.318961
     5    a very negative customer review 0.201915


In [23]:
import pandas as pd

df = pd.read_csv("../data/yelp_sample_1000.csv")


def test_label_mapping():

    assert set(df["label"].unique()) == {0, 1, 2, 3, 4}

    assert (df["stars"] == df["label"] + 1).all()

    assert df["stars"].between(1, 5).all()

    print("Test 1 — Label Mapping: PASSED")


def test_balanced_sample():

    star_counts = df["stars"].value_counts().sort_index()

    assert star_counts.to_dict() == {
        1: 200,
        2: 200,
        3: 200,
        4: 200,
        5: 200
    }

    print("Test 2 — Balanced Sample: PASSED")


def test_valid_text():

    assert df["text"].apply(lambda x: isinstance(x, str)).all()

    assert df["text"].apply(lambda x: bool(x.strip())).all()

    print("Test 3 — Valid Text: PASSED")


test_label_mapping()
test_balanced_sample()
test_valid_text()

Test 1 — Label Mapping: PASSED
Test 2 — Balanced Sample: PASSED
Test 3 — Valid Text: PASSED
